# Fact-Checking Pipeline — full run on Colab

```
raw text ─▶ 🟦 TASK 1  Claim Detection   (RoBERTa, fine-tuned on ClaimBuster)
         ─▶ 🟨 TASK 2  Evidence Retrieval (BM25 → bi-encoder → cross-encoder)
         ─▶ 🟥 TASK 3  Verification       (DeBERTa NLI → SUPPORTS / REFUTES / NOT ENOUGH INFO)
```

**Before running**
1. **Runtime → Change runtime type → T4 GPU**.
2. Make sure the branch in the config cell below is pushed to GitHub. The notebook clones the code from there.
3. **Runtime → Run all**. The first run downloads FEVER (~1.7 GB) and builds the corpus (~30 min).
   Later runs skip everything that is already in Drive.

**Everything is stored in Google Drive** under `MyDrive/nlp-final-project/`, so it survives Colab restarts:

| Drive folder | Contents |
|---|---|
| `models/` | trained Task 1 model (`task1_roberta-base_label/model`) |
| `hf_cache/` | pretrained Hugging Face models (NLI DeBERTa, MiniLM encoders, roberta-base) |
| `data/` | FEVER claims, `wiki-pages.zip`, the Task 2 corpus subset, Task 1 splits |
| `outputs/` | evaluation metrics and predictions for Task 2 and Task 3 |

The project's `data/`, `models/` and `outputs/` folders are linked to these Drive folders, so every
script reads from and writes to Drive directly.

Cells marked **(optional)** are evaluations or training that you can skip for a quick demo.

# ⚙️ SETUP

### Config

In [ ]:
REPO_URL  = 'https://github.com/Amzar123/nlp-final-project.git'
BRANCH    = 'task3-nli-verification'   # branch to clone; must be pushed to GitHub
DRIVE_DIR = '/content/drive/MyDrive/nlp-final-project'
PROJECT_DIR = '/content/project'

import os
# Pretrained Hugging Face models are cached in Drive, so they are downloaded only once.
# Must be set before transformers is imported.
os.environ['HF_HOME'] = f'{DRIVE_DIR}/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

### GPU check

In [ ]:
!nvidia-smi -L
import torch
print('cuda available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

for name in ['data', 'models', 'outputs', 'hf_cache']:
    os.makedirs(f'{DRIVE_DIR}/{name}', exist_ok=True)
!ls -la "{DRIVE_DIR}"

### Get the code + link the Drive folders
Re-running is safe: the code is updated to the latest commit of `BRANCH`, and the Drive folders are untouched.

In [ ]:
import subprocess, sys

def sh(cmd):
    # Run a shell command with live output; stop the notebook if it fails.
    ip = get_ipython()
    ip.system(cmd)
    if ip.user_ns.get('_exit_code', 0) != 0:
        raise RuntimeError(f'Command failed: {cmd}')

if not os.path.exists(f'{PROJECT_DIR}/.git'):
    sh(f'git clone -q -b {BRANCH} {REPO_URL} {PROJECT_DIR}')
else:
    sh(f'git -C {PROJECT_DIR} fetch -q origin {BRANCH} && git -C {PROJECT_DIR} reset -q --hard origin/{BRANCH}')
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# data/, models/, outputs/ in the project -> Drive
for name in ['data', 'models', 'outputs']:
    link, target = f'{PROJECT_DIR}/{name}', f'{DRIVE_DIR}/{name}'
    if os.path.islink(link):
        continue
    if os.path.exists(link):
        raise RuntimeError(f'{link} exists and is not a link to Drive; remove it and re-run')
    os.symlink(target, link)

sh('git log -1 --oneline')
sh('ls -la data models outputs')

### Install dependencies

In [ ]:
# Colab already has a CUDA build of torch: keep it, install the rest.
sh("grep -v -E '^torch' requirements.txt > /tmp/requirements.txt && pip install -q -r /tmp/requirements.txt")
sh('python -m spacy download en_core_web_sm 2>&1 | tail -1')

import transformers, sentence_transformers
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| sentence-transformers', sentence_transformers.__version__)

In [ ]:
# Prints a clear marker in the cell output for each pipeline stage
def stage(title):
    line = '━' * 90
    print(f'\n{line}\n  {title}\n{line}')

# 📦 DATA
FEVER claims + Wikipedia corpus subset for Task 2 and Task 3. Skipped if it already exists in Drive.

`build_corpus` downloads `shared_task_dev.jsonl` and `wiki-pages.zip` (1.7 GB) into `data/raw/`,
then writes `data/processed/wiki_subset.jsonl` and `claims_sample.jsonl`. First run: ~20–30 min.

In [ ]:
if os.path.exists('data/processed/wiki_subset.jsonl') and os.path.exists('data/processed/claims_sample.jsonl'):
    print('Corpus already in Drive, skipping.')
else:
    sh('python -m src.task2_evidence_retrieval.build_corpus --n-claims 500 --n-random 20000')
sh('ls -lh data/processed')

# 🟦 TASK 1 · Claim Detection
Splits raw text into sentences and keeps the ones that are factual claims.

### 1.1 Get the model from Drive
Looks for `models/task1_roberta-base_label/model` in Drive. If only the zip from `task1_colab.ipynb`
(`task1_roberta-base_label.zip`) is there, it is extracted. If neither exists, set
`TRAIN_IF_MISSING = True` to train it here (~15 min on a T4).

In [ ]:
TRAIN_IF_MISSING = True

TASK1_RUN = 'task1_roberta-base_label'
TASK1_MODEL_DIR = f'models/{TASK1_RUN}/model'
TASK1_ZIP = f'{DRIVE_DIR}/{TASK1_RUN}.zip'

if os.path.exists(f'{TASK1_MODEL_DIR}/model.safetensors'):
    print('Task 1 model found in Drive.')
elif os.path.exists(TASK1_ZIP):
    print(f'Extracting {TASK1_ZIP} ...')
    sh(f'unzip -oq "{TASK1_ZIP}" -d "{DRIVE_DIR}"')   # the zip contains models/{TASK1_RUN}/...
elif TRAIN_IF_MISSING:
    print('No Task 1 model in Drive, training one (saved straight to Drive) ...')
    sh('python -m src.data_prep.download --no-wiki')
    sh('python -m src.data_prep.build_task1_data')
    sh('python -m src.task1_claim_detection.train --model roberta-base --label-col label --batch-size 32 --grad-accum 1')
else:
    raise FileNotFoundError(f'No Task 1 model in {DRIVE_DIR}/models and TRAIN_IF_MISSING is False')

sh(f'ls -lh {TASK1_MODEL_DIR}')
sh(f'cat {TASK1_MODEL_DIR}/threshold.json')

### 1.2 Load + try it

In [ ]:
from src.task1_claim_detection.infer import ClaimDetector

detector = ClaimDetector(TASK1_MODEL_DIR)
print('device:', detector.device, '| threshold:', detector.threshold)

stage('🟦 TASK 1 · Claim Detection — demo')
demo_text = 'Hello everyone! I think pizza is great. Barack Obama was born in Hawaii in 1961.'
for r in detector.detect(demo_text, return_all=True):
    print(f"{'CLAIM' if r['is_claim'] else '  -  '}  p_claim={r['p_claim']:.3f}  {r['sentence']}")

# 🟨 TASK 2 · Evidence Retrieval & Reranking
BM25 picks the top-5 pages, a bi-encoder picks the top-30 sentences, and a cross-encoder reranks them to the top-5.

**Note:** the corpus is a subset built for 500 FEVER claims. A claim outside that sample will usually
not find its evidence, so the demo uses a claim from `claims_sample.jsonl`.

### 2.1 Load + try it

In [ ]:
from src.task2_evidence_retrieval.pipeline import EvidencePipeline
from src.task2_evidence_retrieval.data import read_claims_cache

evidence_pipe = EvidencePipeline.from_corpus_file('data/processed/wiki_subset.jsonl')
sample_claims = read_claims_cache('data/processed/claims_sample.jsonl')
print(f'{len(evidence_pipe.retriever.pages):,} pages in corpus, {len(sample_claims)} sample claims')

DEMO_CLAIM = sample_claims[0]
stage('🟨 TASK 2 · Evidence Retrieval — demo')
print(f"Claim: {DEMO_CLAIM['claim']}   (FEVER label: {DEMO_CLAIM['label']})\n")
demo_evidence = evidence_pipe.run(DEMO_CLAIM['claim'], top_k=5)
for i, ev in enumerate(demo_evidence, 1):
    print(f"{i}. [{ev['title']} #{ev['line_id']}] rerank={ev['rerank_score']:.2f}\n   {ev['text']}")

### 2.2 Evaluation (optional) → `outputs/task2/`

In [ ]:
sh('python -m src.task2_evidence_retrieval.evaluate')   # add --limit 50 for a quick test

# 🟥 TASK 3 · Verification (NLI)
DeBERTa NLI scores each (evidence, claim) pair and aggregates them into SUPPORTS / REFUTES / NOT ENOUGH INFO.

### 3.1 Load + try it (on the Task 2 demo evidence)

In [ ]:
from src.task3_verification.nli import NLIVerifier

verifier = NLIVerifier()   # MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli, threshold 0.5
print('device:', verifier.device)

stage('🟥 TASK 3 · Verification — demo')
res = verifier.verify(DEMO_CLAIM['claim'], demo_evidence)
print(f"Claim:   {DEMO_CLAIM['claim']}")
print(f"Verdict: {res['label']} ({res['confidence']:.3f})   gold: {DEMO_CLAIM['label']}\n")
for i, ev in enumerate(res['evidence'], 1):
    nli = ev['nli']
    print(f"{i}. E={nli['entailment']:.2f} C={nli['contradiction']:.2f}  [{ev['title']}] {ev['text']}")

### 3.2 Evaluation (optional) → `outputs/task3/`
`retrieved` = real pipeline evidence; `oracle` = gold evidence (upper bound for the NLI step).

In [ ]:
sh('python -m src.task3_verification.evaluate')                     # add --limit 50 for a quick test

In [ ]:
sh('python -m src.task3_verification.evaluate --oracle-evidence')

# 🔗 FULL PIPELINE · Task 1 → Task 2 → Task 3
Type any text in the form. Leave it empty to use a demo text built around a FEVER claim from the corpus.
Every claim result is also saved to `outputs/pipeline/last_run.json` in Drive.

In [ ]:
TEXT = ''  # @param {type:"string"}
TOP_K = 5  # @param {type:"integer"}

from src.pipeline import FactCheckPipeline

pipe = FactCheckPipeline(claim_detector=detector, evidence_pipeline=evidence_pipe, verifier=verifier, top_k=TOP_K)
text = TEXT.strip() or f"Good evening everyone, thanks for coming. {sample_claims[1]['claim']} I really love pizza here."
print('Input:', text)

stage('🟦 TASK 1 · Claim Detection')
sentences = detector.detect(text, return_all=True)
for r in sentences:
    print(f"{'CLAIM' if r['is_claim'] else '  -  '}  p_claim={r['p_claim']:.3f}  {r['sentence']}")
claims = [r for r in sentences if r['is_claim']]
if not claims:
    print('\nNo factual claims found, nothing to verify.')

results = []
for n, c in enumerate(claims, 1):
    r = pipe.verify_claim(c['sentence'], c['p_claim'])   # Task 2 + Task 3
    results.append(r)

    stage(f"🟨 TASK 2 · Evidence  —  claim {n}/{len(claims)}: {r['claim']}")
    for i, ev in enumerate(r['evidence'], 1):
        print(f"{i}. [{ev['title']} #{ev['line_id']}] rerank={ev['rerank_score']:.2f}  {ev['text']}")

    stage(f"🟥 TASK 3 · Verdict  —  claim {n}/{len(claims)}")
    print(f"{r['verdict']}  (confidence {r['verdict_confidence']:.3f})")
    for i, ev in enumerate(r['evidence'], 1):
        print(f"  {i}. E={ev['nli']['entailment']:.2f} C={ev['nli']['contradiction']:.2f}  {ev['title']}")

import json
os.makedirs('outputs/pipeline', exist_ok=True)
with open('outputs/pipeline/last_run.json', 'w') as f:
    json.dump({'text': text, 'results': results}, f, indent=2, ensure_ascii=False, default=float)
print('\nSaved to outputs/pipeline/last_run.json')

# 📊 RESULTS SUMMARY
Reads the metrics files in Drive (only those you have run).

In [ ]:
import glob, json

stage('🟦 TASK 1 · Claim Detection (ClaimBuster test set)')
for path in sorted(glob.glob('models/task1_*/metrics.json')):
    m = json.load(open(path))
    if 'test@tuned' in m:
        t = m['test@tuned']
        print(f"{m['run']:<40} thr={m['tuned_threshold']:.2f}  macroF1={t['macro_f1']:.4f}  "
              f"P={t['precision']:.4f}  R={t['recall']:.4f}  AUC={t['roc_auc']:.4f}")

stage('🟨 TASK 2 · Evidence Retrieval')
path = 'outputs/task2/metrics.json'
if os.path.exists(path):
    m = json.load(open(path))
    for k, v in m.items():
        if isinstance(v, float):
            print(f'  {k:<28} {v:.4f}')
else:
    print('  not run yet (section 2.2)')

stage('🟥 TASK 3 · Verification')
for tag in ['retrieved', 'oracle']:
    path = f'outputs/task3/metrics_{tag}.json'
    if not os.path.exists(path):
        print(f'  {tag:<10} not run yet (section 3.2)')
        continue
    m = json.load(open(path))
    for name, key in [('fixed thr', 'fixed_threshold'), (f"tuned thr={m['tuned_threshold']:.2f}", 'tuned_test_half')]:
        r = m[key]
        print(f"  {tag:<10} {name:<15} acc={r['accuracy']:.4f}  macroF1={r['macro_f1']:.4f}  FEVER={r['fever_score']:.4f}")